# Strategy research — group MaRs-777 (Thief)

**This notebook explains and displays. It computes nothing of its own.**
Every statistic comes from tested functions in `research/` and is read from
result files committed when the experiment ran. It runs no games, touches no
network and needs no credential.

**The finding it reports is a decision not to change anything.** That is a
real research outcome, and it is presented with the evidence that supports
it rather than as an absence of work.


## 0. Reproducing this notebook

```bash
uv sync --group notebook
uv run jupyter nbconvert --execute --to notebook --inplace \
    notebooks/strategy_research.ipynb
```

**Jupyter is deliberately not in the committed lockfile** - a tournament
agent must not need a notebook stack to play a game. The tables and figures
regenerate without it:

```bash
uv run python -m research.bench_main analyse --out results
```


In [ ]:
import csv
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
overall = json.loads((RESULTS / "tables" / "overall.json").read_text(encoding="utf-8"))
print("role            :", overall["baseline"]["role"])
print("strategy        :", overall["baseline"]["strategy"])
print("statistical unit:", overall["statistical_unit"])
print("headline N      :", overall["headline_scenarios"])

## 1. What was measured

The same benchmark laboratory the police repository uses: seven opponent
families x six source-legal configuration families x three disjoint seed
banks, scored per **unique scenario**, with a deterministic bootstrap for
every interval.

The thief's primary metric is survival-driven `win_rate`.


In [ ]:
head = overall["overall"]["win_rate"]
print(f"thief win rate  : {head['mean']:.4f}  [{head['ci_low']:.4f}, {head['ci_high']:.4f}]")
print()
rows = list(csv.DictReader((RESULTS / "tables" / "by_opponent_family.csv").open()))
for one in sorted(rows, key=lambda r: float(r["win_rate"])):
    print(f"  {one['group']:<20} {float(one['win_rate']):.4f}  N={one['n']}")

## 2. Why the strategy was not changed

**The headline is 0.9886 over 4,988 independent scenarios.** Six of seven
opponent families are at a perfect 1.0000; only the two barrier-using
families - `adversarial_corner` at 0.9497 and `barrier_aware` at 0.9705 -
lose anything at all, on N=713 each.

That is not a policy with room to optimise. It is a policy near the ceiling
of what this corpus can measure, where the remaining losses come from the
opponent legally shrinking the board rather than from a decision the thief
made badly.

Changing it would have meant spending the risk of a regression to chase at
most 1.1 points of headline, with no mechanism identified for recovering
them. **`NO_CHANGE` was therefore recorded as a result, not as a default**,
and it was re-derived after the Stage-9B-0F methodology correction rather
than inherited from the flawed first measurement.


In [ ]:
for name in ("adversarial_corner", "barrier_aware"):
    one = next(r for r in rows if r["group"] == name)
    print(
        f"{name:<20} {float(one['win_rate']):.4f}  "
        f"[{one['win_rate_ci_low']}, {one['win_rate_ci_high']}]  N={one['n']}"
    )

## 3. This repository ships no barrier policy

`BAR-004` gives barrier placement to the **police alone**. A thief that
disclosed a placement would be judged `ILLEGAL_ACTION` and would lose the
sub-game, so the thief's strategy is movement only - `BaselineStrategy`,
the frozen accessibility-centrality policy.

The competitive barrier research that produced the police repository's
promoted rule is **that** repository's work and is not restated here as
though it were this agent's.


## 4. Why there is no learning curve

**Nothing in this project is trained.** No model, no parameter update, no
epoch, no gradient, no reward. `DOC-001` component (4) asks for learning
curves *if reinforcement learning is used*; none is, so the conditional is
not met and inventing a loss curve would be a fabricated result.


In [ ]:
figures = RESULTS / "figures"
print("committed baseline figures:")
for path in sorted(figures.glob("*.png")):
    print("  ", path.name)

## 5. What this claims

A measured, reproducible description of one deterministic policy against
**our own** opponent corpus. The seven families are models we wrote; the
real opponent is unknown. Nothing here predicts the result of any particular
match.
